In [46]:
# Extract ALFIN city/year observational units that are relevant for our work

import os
import pandas as pd
import numpy as np
import dotenv

import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

dotenv.load_dotenv(dotenv.find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")

INPUT_FILEPATH = os.path.join(MY_DATA_PATH, "raw_data/alfin_raw.parquet")
GEO_FILEPATH = os.path.join(MY_DATA_PATH, "raw_data/alfin_raw_geo.parquet")

OUTPUT_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data/alfin_units.parquet")


UNIT_TYPE_MAP = {
    '0': 'State',
    '1': 'County',
    '2': 'City',
    '3': 'Township',
    '4': 'Special District',
    '5': 'Independent School District or Educational Service Agency'
}

FUNCTION_CODE_MAP = {
    '01': 'Air transportation (airports)', 
    '02': 'Cemeteries',
    '03': 'Miscellaneous commercial activities',
    '04': 'Correctional institutions', 
    '05': 'Other corrections',
    '09': 'Education (school building authorities)',
    '24': 'Fire protection',
    '32': 'Health',
    '40': 'Hospitals',
    '41': 'Industrial development',
    '42': 'Mortgage credit',
    '44': 'Regular highways',
    '45': 'Toll highways',
    '50': 'Housing and community development', 
    '51': 'Drainage', 
    '52': 'Libraries',
    '59': 'Other natural resources',
    '60': 'Parking facilities',
    '61': 'Parks and recreation',
    '62': 'Police protection',
    '63': 'Flood control',
    '64': 'Irrigation',
    '77': 'Public welfare institutions',
    '79': 'Other public welfare',
    '80': 'Sewerage',
    '81': 'Solid waste management',
    '86': 'Reclmaation',
    '87': 'Sea and inland port facilities',
    '88': 'Soil and water conservation',
    '89': 'Other single-function districts',
    '91': 'Water supply utility',
    '92': 'Electric power utility',
    '93': 'Gas supply utility',
    '94': 'Mass transit system utility',
    '96': 'Fire protection and water supply - combination of services',
    '97': 'Natural resources and water supply - combination of services',
    '98': 'Sewerage and water supply - combination of services',
    '99': 'Other multifunction districts'
}


In [47]:
# Load the data

df = pd.read_parquet(INPUT_FILEPATH)
geo_df = pd.read_parquet(GEO_FILEPATH)

# Merge on state names / abbrevs

state_fips_df = pd.read_csv(os.path.join(RAW_DATA_PATH, "state_fips.csv"))
state_fips_df['STATE_FIPS'] = state_fips_df['STATE_FIPS'].astype(str).str.zfill(2)
geo_df = geo_df.merge(state_fips_df[['STATE_FIPS', 'STATE']], on='STATE_FIPS', how='left')

In [48]:
# Keep only counties, cities, townships, and special districts

df = df.loc[df['UNIT_TYPE_CODE'].isin(['1', '2', '3', '4'])]

In [49]:
# Get list of observational units

dfg = df[['ID', 'UNIT_TYPE_CODE']].drop_duplicates()
dfg = dfg.merge(
    geo_df[['ID', 'NAME', 'STATE', 'STATE_FIPS', 'COUNTY_FIPS', 'PLACE_FIPS', 'FUNCTION_CODE']], 
    on=['ID'], how='left'
)
dfg = dfg.drop_duplicates().sort_values(by=['STATE', 'NAME', 'ID']).reset_index(drop=True)



In [50]:
# Map on human readable unit type and functions

dfg['UNIT_TYPE'] = dfg['UNIT_TYPE_CODE'].map(UNIT_TYPE_MAP)
dfg['FUNCTION'] = dfg['FUNCTION_CODE'].map(FUNCTION_CODE_MAP)


In [51]:
# Save file

dfg.to_parquet(OUTPUT_FILEPATH)